In [ ]:
from typing import Dict, List
import os
import requests

def get_priorities_from_slack_node(_: Dict) -> Dict:
    SLACK_BOT_TOKEN = os.getenv("SLACK_BOT_TOKEN")
    CHANNEL_ID = os.getenv("SLACK_CHANNEL_ID")

    headers = {
        "Authorization": f"Bearer {SLACK_BOT_TOKEN}"
    }

    params = {
        "channel": CHANNEL_ID,
        "limit": 50
    }

    response = requests.get("https://slack.com/api/conversations.history", headers=headers, params=params)
    data = response.json()

    if not data.get("ok"):
        raise Exception(f"Slack API error: {data.get('error')}")

    messages = data.get("messages", [])

    # Collect all messages that mention "priority" or "priorities"
    priority_candidates = []
    for msg in messages:
        text = msg.get("text", "")
        if "priority" in text.lower():
            priority_candidates.append(text.strip())

    return {"raw_priority_messages": priority_candidates}


In [ ]:
from slack_sdk import WebClient
import os
from dotenv import load_dotenv
load_dotenv()
client = WebClient(token=os.getenv("SLACK_BOT_TOKEN"))
channel_to_listen=os.getenv("SLACK_CHANNEL_ID")
response = client.auth_test()
print("Bot User ID:", response["user_id"])


Bot User ID: U093VRW6Y84


In [2]:
from slack_sdk import WebClient
import os
from dotenv import load_dotenv
load_dotenv()
client = WebClient(token=os.getenv("SLACK_BOT_TOKEN"))
channel_to_listen=os.getenv("SLACK_CHANNEL_ID")

response=client.conversations_history(channel=channel_to_listen,limit=10)
messages=response['messages']

In [3]:
messages

[{'user': 'U093VRW6Y84',
  'type': 'message',
  'ts': '1751605489.182639',
  'bot_id': 'B093VRW4CN8',
  'app_id': 'A093P86G738',
  'text': '*Daily Update:*\n```Daily Update (4th July 2025 (Friday))\n1. New Haven – 75%\n2. Bioforce – 0%\n3. Canal – --%\n4. Aptar – 60%\n5. Sandbox – 27%\n6. Bankwell – 33%\n7. Mudflap – 33%\n8. Hylant JV – --%\n9. Lifford – 12%\n10. DevOps &amp; Support – 14%```',
  'team': 'T08B1A4E0KZ',
  'bot_profile': {'id': 'B093VRW4CN8',
   'deleted': False,
   'name': 'Demo App',
   'updated': 1751362885,
   'app_id': 'A093P86G738',
   'user_id': 'U093VRW6Y84',
   'icons': {'image_36': 'https://a.slack-edge.com/80588/img/plugins/app/bot_36.png',
    'image_48': 'https://a.slack-edge.com/80588/img/plugins/app/bot_48.png',
    'image_72': 'https://a.slack-edge.com/80588/img/plugins/app/service_72.png'},
   'team_id': 'T08B1A4E0KZ'},
  'blocks': [{'type': 'rich_text',
    'block_id': 'n4nav',
    'elements': [{'type': 'rich_text_section',
      'elements': [{'type': '

In [8]:
for i in messages:
    timestamp=i['ts']
    content=i['text']
    print(timestamp + " "+content)
    

1751605489.182639 *Daily Update:*
```Daily Update (4th July 2025 (Friday))
1. New Haven – 75%
2. Bioforce – 0%
3. Canal – --%
4. Aptar – 60%
5. Sandbox – 27%
6. Bankwell – 33%
7. Mudflap – 33%
8. Hylant JV – --%
9. Lifford – 12%
10. DevOps &amp; Support – 14%```
1751538569.735969 asdasd
1751538565.451129 asdasd
1751536306.690949 *Daily Update:*
```Daily Update (3rd July 2025 (Thursday))
1. Trident Gateway – --%
2. Elara Health – --%
3. Nimbus Flow – --%
4. Orbit Analytics – --%
5. Fission Cloud – --%
6. New Haven – 75%
7. Bioforce – 0%
8. Canal – --%
9. Aptar – 60%
10. Sandbox – 24%
11. Bankwell – 16%
12. Mudflap – 33%
13. Hylant JV – --%
14. Lifford – 12%
15. DevOps &amp; Support – 14%```
1751534980.102149 Hey team, our focus this week should be:
1. Trident Gateway
2. Elara Health
3. Nimbus Flow
4. Orbit Analytics
5. Fission Cloud
1751534238.777049 the tasks we need to check this week are the following: New Haven, FakeDB, NewDB.
1751533724.663819 *Daily Update:*
```Daily Update (3rd J

In [ ]:
workflow.add_node("llm_extract_priorities", lambda state: llm_extract_priorities_node(state, llm))
workflow.add_edge("get_priorities_from_slack", "llm_extract_priorities")


In [2]:
if __name__ == "__main__":
    from dotenv import load_dotenv
    load_dotenv()

    llm = ChatOpenAI(temperature=0, model="gpt-4o-mini")  # or ChatGroq, Gemini, Claude etc.
    graph = build_priority_flow(llm)

    final_output = graph.invoke({})
    print("\n✅ Final extracted priorities:\n")
    for p in final_output["priorities"]:
        print(f"- {p}")


/tmp/ipykernel_15606/3107129786.py:5: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(temperature=0, model="gpt-4o-mini")  # or ChatGroq, Gemini, Claude etc.



✅ Final extracted priorities:

- Update the email for workflow
- Develop HubSpot to AD integration
- Lead import Template Updates
- Subscriptions Types requirements


In [3]:
print(final_output)

{'raw_priority_messages': ['PRIORITIES for this week are:\nUpdate the email for workflow\nDevelop Hubsppot to AD integration\nLead import Template Updates\nSubscriptions Types requirements', 'PRIORITIES for this week are:', 'PRIORITIES for this week are:'], 'priorities': ['Update the email for workflow', 'Develop HubSpot to AD integration', 'Lead import Template Updates', 'Subscriptions Types requirements'], 'raw_llm_output': '1. Update the email for workflow  \n2. Develop HubSpot to AD integration  \n3. Lead import Template Updates  \n4. Subscriptions Types requirements  '}


In [1]:
import os
import requests
import asana
ASANA_TOKEN = os.getenv("ASANA_ACCESS_TOKEN")

client=asana.Client.access_token(ASANA_TOKEN)



In [2]:
from dotenv import load_dotenv
load_dotenv()


True

In [3]:
import asana
import os
from typing import List, Dict

def get_asana_completion_report(priorities: List[str]) -> Dict[str, Dict]:
    # Setup Asana client
    ASANA_TOKEN = os.getenv("ASANA_ACCESS_TOKEN")
    if not ASANA_TOKEN:
        raise ValueError("Missing ASANA_ACCESS_TOKEN in environment.")

    client = asana.Client.access_token(ASANA_TOKEN)
    client.options['page_size'] = 100

    # Step 1: Get workspaces and pick the first one (or filter by name)
    workspaces = client.workspaces.get_workspaces()
    WORKSPACE_GID = "your_workspace_gid_here"

    
    # Step 2: Find project ID for "Agents"
    projects = client.projects.find_by_workspace(workspace_gid, {"archived": False})
    project_id = None
    for project in projects:
        if project["name"].lower() == "agents":
            project_id = project["gid"]
            break
    if not project_id:
        raise ValueError("Project 'Agents' not found.")

    # Step 3: Get sections from the project
    sections = client.sections.find_by_project(project_id)
    section_map = {section["name"].lower(): section["gid"] for section in sections}

    # Step 4: For each priority, match to section, get tasks
    report = {}

    for priority in priorities:
        section_name = priority.strip().lower()
        section_id = section_map.get(section_name)

        if not section_id:
            report[priority] = {"completed": 0, "total": 0, "percent": 0.0, "status": "Section not found"}
            continue

        tasks = list(client.tasks.find_by_section(section_id))
        total = len(tasks)
        completed = sum(1 for task in tasks if task.get("completed", False))

        percent = round((completed / total) * 100, 2) if total else 0.0

        report[priority] = {
            "completed": completed,
            "total": total,
            "percent": percent,
            "status": "ok"
        }

    return report


In [4]:

def get_completion_report_for_all_projects(workspace_gid: str):
    report = {}

    # Step 1: Get all projects
    projects = client.projects.get_projects_for_workspace(
        workspace_gid, {"archived": False, "opt_fields": "name,gid"}
    )

    for project in projects:
        project_name = project["name"]
        project_id = project["gid"]

        # Step 2: Get all tasks in this project
        try:
            tasks = list(client.tasks.find_by_project(project_id, {"opt_fields": "completed"}))
            total = len(tasks)
            completed = sum(1 for task in tasks if task.get("completed", False))
            percent = round((completed / total) * 100, 2) if total else 0.0

            report[project_name] = {
                "completed": completed,
                "total": total,
                "percent": percent
            }
        except Exception as e:
            print(f"❌ Error fetching tasks for {project_name}: {e}")
            report[project_name] = {
                "completed": 0,
                "total": 0,
                "percent": 0.0,
                "error": str(e)
            }

    return report

In [6]:
priorities = ["Agents", "Full Stack", "Design"]
workspace_gid="1208832380740289"
report = get_completion_report_for_all_projects(workspace_gid=workspace_gid)

for section, stats in report.items():
    print(f"{section}: {stats['completed']}/{stats['total']} completed ({stats['percent']}%)")


ValueError: Missing access token.

In [7]:

workspace_gid='1208832380740289'
projects = client.projects.get_projects_for_workspace(
        workspace_gid,
        params={"archived": False, "opt_fields": "name,gid"}
    )

In [8]:
for project in projects:
    project_name=project["name"]
    project_id=project['gid']
    print(project)
    #print(project_name)

ValueError: Missing access token.

In [10]:
def list_asana_workspaces():
    ASANA_TOKEN = os.getenv("ASANA_ACCESS_TOKEN")
    client = asana.Client.access_token(ASANA_TOKEN)
    workspaces = client.workspaces.get_workspaces()

    print("🔍 Available Asana Workspaces:")
    for ws in workspaces:
        print(f"- {ws['name']} (gid: {ws['gid']})")


In [11]:
list_asana_workspaces()

🔍 Available Asana Workspaces:
- workabot.ai (gid: 1208837199882893)
- Workabot.ai (gid: 1208832380740289)


/home/abhishek/daily/tasks/langchain/lang-env/lib/python3.10/site-packages/asana/client.py:156: UserWarning: This request is affected by the "new_goal_memberships" deprecation. Please visit this url for more info: https://forum.asana.com/t/launched-team-sharing-for-goals/378601
Adding "new_goal_memberships" to your "Asana-Enable" or "Asana-Disable" header will opt in/out to this deprecation and suppress this warning.
  warnings.warn(message)
